# 02 — XGBoost 단일 회귀

Anchor ±50% Narrow HPO + 후처리 매트릭스. (1차 reg_only 데이터 없음 — Y>0 우회 anchor 사용)

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/02_reg_single/xgb/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` ([strategy_common.md §1](../strategy_common.md))
- **HPO**: 150 trial, anchor 첫 trial enqueue ([strategy.md §5.5](strategy.md), Y>0 컨텍스트에서 fork — reg_only 1차 산출물 없음), narrow ±50% ([§7.5](strategy.md))
- **손실함수**: `reg:squarederror / count:poisson / reg:tweedie` 3종 ([§6](strategy.md))
- **target transform**: `'none'` 고정 (strategy_common §24 — log1p_check 검증)

## 모듈 의존성 ([strategy.md §13](strategy.md))

1. `3_modeling/modules/` 이관 완료
2. `models.py` — `xgb_space`에 `count:poisson` 추가, `reg:tweedie_*` categorical → `reg:tweedie` + `tweedie_variance_power=suggest_float(1.05, 1.95)` 통합
3. `hpo.py` — `enqueue_trials` 인자 + trial 내 target_transform 분기 (`reg:tweedie` 시 OFF)
4. `postprocess.py` — `Q25/Q75` + `zero_clip_space='log'` 분기

## 결정 필요 (strategy.md §14)



## 1. 환경 설정 + 모듈 import

In [ ]:
import os, sys

# ── Google Drive 파일 ID (Colab 사용 시. 로컬은 무시됨) ──
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip (utils/, setup.py)
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip (CSV 4개)
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip (cleaning/outlier/scaling/...)
GDRIVE_MODELING_ID      = ''   # ★ modules.zip Drive ID (3_modeling/modules/) — 추후 입력

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from modules import preprocess, hpo, models   # noqa: E402

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

## 2. 실험 설정

In [ ]:
MODEL_NAME = 'xgb'
EXP_ID     = f'reg-{MODEL_NAME}-002'
EXP_MEMO   = 'Anchor ±50% Narrow (Y>0 우회 base, 1차 reg_only 데이터 없음)'
USER       = 'jh'

N_TRIALS = 150
N_FOLDS  = 5
N_STARTUP_TRIALS = 10
N_JOBS   = 7

TARGET_TRANSFORM = 'none'  # ★ strategy_common §24 (log1p_check 검증: none=log1p 동등)
CLIP_Y_EXTREME   = True

OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# anchor (strategy.md §5.5, Y>0 컨텍스트 우회 base)
XGB_ANCHOR = {
    'objective':              'reg:tweedie',  # categorical: reg:squarederror | count:poisson | reg:tweedie
    'tweedie_variance_power': 1.5,
    'n_estimators':           1423,
    'learning_rate':          0.0363,
    'max_depth':              10,
    'min_child_weight':       0.621,
    'subsample':              0.728,
    'colsample_bytree':       0.618,
    'reg_alpha':              0.01680,
    'reg_lambda':             3.890e-06,
    'gamma':                  3.837e-06,
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + target clip + transform

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# Target transform: 'none' 고정 (strategy_common §24 — log1p_check 검증: none=log1p 동등)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24 — 트리 target_transform=none 통일)')

## 4. 전처리 (PP_FIXED 고정)

In [ ]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

## 5. Optuna HPO (anchor 첫 trial enqueue + narrow ±50%)

[strategy.md §7.5](strategy.md): 1차 reg_only 데이터 없어 anchor ±50% narrow. xgb_space는 §13 #2 변경 후 `count:poisson` 후보를 포함해야 함.

In [ ]:
study_meta_for_save = {
    'exp_id':              EXP_ID,
    'exp_memo':            EXP_MEMO,
    'user':                USER,
    'model_name':          MODEL_NAME,
    'target_transform':    TARGET_TRANSFORM,
    'clip_y_extreme':      CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'n_jobs':              N_JOBS,
    'n_startup_trials':    N_STARTUP_TRIALS,
    'seed_kfold':          SEED,
    'anchor':              XGB_ANCHOR,
}

# strategy_common.md §4·§5: anchor enqueue → study 사전 생성 후 run_hpo 가 resume
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
_study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=False,
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
    pruner=MedianPruner(n_warmup_steps=10),
)
hpo.enqueue_anchor(_study, XGB_ANCHOR)

res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=True,
    user_attrs=study_meta_for_save,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study                 = res['study']
best_params_for_refit = res['best_params']
study_meta_for_save['hpo_best_value'] = float(res['best_value'])

# 검증 (strategy.md §15): anchor가 trial 0 인지
first_trial_params = study.trials[0].params
anchor_keys_present = {k: first_trial_params.get(k) for k in XGB_ANCHOR if k in first_trial_params}
print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'[검증] trial 0 params (anchor 키만): {anchor_keys_present}')
print(f'best_params = {best_params_for_refit}')

## 6. Best trial 재학습 (K-fold OOF)

In [ ]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

print(f'\n[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 7. 후처리 매트릭스 + 산출물 저장

In [ ]:
POSTPROCESS_CONFIG = {
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
    'use_pi_threshold': False,
}

hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass